<a href="https://colab.research.google.com/github/pplateena/AIIIT-NULP/blob/main/l3/evolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота №3
## Використання еволюційних алгоритмів

**Варіант №7**

### Еволюційні оператори:
- **Відбір**: Ранжування (Ranking Selection)
- **Схрещування**: Багатоточкове (Multi-point Crossover)
- **Мутація**: Випадкове скидання (Random Reset Mutation)

### Задачі для дослідження:
1. **Задача мінімізації функції Розенброка** (діапазон [-5, 5])
2. **Задача рюкзака**

### Мета:
Порівняти ефективність різних еволюційних операторів та провести аналіз результатів експериментів.

In [ ]:
# Імпортуємо необхідні бібліотеки
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Optional
import random
import time
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Встановлюємо seeds для відтворюваності
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print("✅ Бібліотеки імпортовано успішно")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Pandas: {pd.__version__}")

# Перевіряємо доступність GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Використовується пристрій: {device}")

## 1. Основний клас еволюційного алгоритму

Реалізуємо базовий еволюційний алгоритм з вказаними операторами.

In [ ]:
@dataclass
class EvolutionConfig:
    """Конфігурація еволюційного алгоритму"""
    population_size: int = 100
    generations: int = 200
    crossover_rate: float = 0.8
    mutation_rate: float = 0.1
    elitism: int = 2  # Кількість найкращих особин, що переходять без змін
    selection_pressure: float = 2.0  # Тиск відбору для ранжування
    
    def __str__(self):
        return f"""Конфігурація EA:
        - Розмір популяції: {self.population_size}
        - Покоління: {self.generations}
        - Ймовірність схрещування: {self.crossover_rate}
        - Ймовірність мутації: {self.mutation_rate}
        - Елітизм: {self.elitism}
        - Тиск відбору: {self.selection_pressure}"""

print("✅ Конфігурацію створено")

In [ ]:
class EvolutionaryAlgorithm:
    """Базовий еволюційний алгоритм з варіантом №7"""
    
    def __init__(self, config: EvolutionConfig, problem_dim: int, bounds: Tuple[float, float]):
        self.config = config
        self.problem_dim = problem_dim
        self.bounds = bounds
        self.population = None
        self.fitness_values = None
        self.history = {
            'best_fitness': [],
            'avg_fitness': [],
            'worst_fitness': [],
            'best_individual': []
        }
    
    def initialize_population(self) -> np.ndarray:
        """Ініціалізація популяції"""
        low, high = self.bounds
        population = np.random.uniform(
            low, high, 
            (self.config.population_size, self.problem_dim)
        )
        return population

print("✅ Базовий клас створено")

In [ ]:
# Додаємо метод ранжування до класу EvolutionaryAlgorithm
def ranking_selection(self, fitness_values: np.ndarray, k: int = None) -> List[int]:
    """Ранжування як метод відбору (Variant #7)"""
    if k is None:
        k = self.config.population_size
        
    # Сортуємо індекси за фітнесом (для мінімізації - за зростанням)
    sorted_indices = np.argsort(fitness_values)
    
    # Присвоюємо ранги (найкращий = найвищий ранг)
    ranks = np.zeros(len(fitness_values))
    for i, idx in enumerate(sorted_indices):
        ranks[idx] = len(fitness_values) - i
    
    # Лінійне ранжування з тиском відбору
    s = self.config.selection_pressure
    n = len(fitness_values)
    probabilities = np.zeros(n)
    
    for i in range(n):
        rank_i = ranks[i]
        probabilities[i] = (2 - s) / n + (2 * rank_i * (s - 1)) / (n * (n - 1))
    
    # Нормалізуємо ймовірності
    probabilities = probabilities / probabilities.sum()
    
    # Відбираємо особини
    selected_indices = np.random.choice(
        len(fitness_values), 
        size=k, 
        p=probabilities, 
        replace=True
    )
    
    return selected_indices.tolist()

# Додаємо метод до класу
EvolutionaryAlgorithm.ranking_selection = ranking_selection

print("✅ Ранжування (Variant #7) додано")

In [ ]:
# Додаємо багатоточкове схрещування до класу EvolutionaryAlgorithm
def multi_point_crossover(self, parent1: np.ndarray, parent2: np.ndarray, 
                         n_points: int = 2) -> Tuple[np.ndarray, np.ndarray]:
    """Багатоточкове схрещування (Variant #7)"""
    if len(parent1) < n_points + 1:
        n_points = max(1, len(parent1) - 1)
    
    # Вибираємо точки схрещування
    crossover_points = sorted(np.random.choice(
        range(1, len(parent1)), 
        size=n_points, 
        replace=False
    ))
    
    child1 = parent1.copy()
    child2 = parent2.copy()
    
    # Міняємо місцями сегменти між точками схрещування
    swap = False
    start_idx = 0
    
    for end_idx in crossover_points + [len(parent1)]:
        if swap:
            child1[start_idx:end_idx] = parent2[start_idx:end_idx]
            child2[start_idx:end_idx] = parent1[start_idx:end_idx]
        swap = not swap
        start_idx = end_idx
    
    return child1, child2

# Додаємо метод до класу
EvolutionaryAlgorithm.multi_point_crossover = multi_point_crossover

print("✅ Багатоточкове схрещування (Variant #7) додано")

In [ ]:
# Додаємо випадкове скидання до класу EvolutionaryAlgorithm
def random_reset_mutation(self, individual: np.ndarray) -> np.ndarray:
    """Випадкове скидання як мутація (Variant #7)"""
    mutated = individual.copy()
    
    for i in range(len(individual)):
        if np.random.random() < self.config.mutation_rate:
            # Повністю замінюємо значення на випадкове в допустимих межах
            low, high = self.bounds
            mutated[i] = np.random.uniform(low, high)
    
    return mutated

# Додаємо метод до класу
EvolutionaryAlgorithm.random_reset_mutation = random_reset_mutation

print("✅ Випадкове скидання (Variant #7) додано")

In [ ]:
# Додаємо основні методи еволюції до класу EvolutionaryAlgorithm
def evolve_generation(self, fitness_func):
    """Еволюція одного покоління"""
    # Обчислюємо фітнес
    self.fitness_values = np.array([fitness_func(ind) for ind in self.population])
    
    # Зберігаємо статистику
    best_idx = np.argmin(self.fitness_values)  # Для мінімізації
    self.history['best_fitness'].append(self.fitness_values[best_idx])
    self.history['avg_fitness'].append(np.mean(self.fitness_values))
    self.history['worst_fitness'].append(np.max(self.fitness_values))
    self.history['best_individual'].append(self.population[best_idx].copy())
    
    # Створюємо нову популяцію
    new_population = []
    
    # Елітизм: зберігаємо найкращих
    if self.config.elitism > 0:
        elite_indices = np.argsort(self.fitness_values)[:self.config.elitism]
        for idx in elite_indices:
            new_population.append(self.population[idx].copy())
    
    # Генеруємо решту популяції
    while len(new_population) < self.config.population_size:
        # Відбір батьків
        parent_indices = self.ranking_selection(self.fitness_values, k=2)
        parent1 = self.population[parent_indices[0]]
        parent2 = self.population[parent_indices[1]]
        
        # Схрещування
        if np.random.random() < self.config.crossover_rate:
            child1, child2 = self.multi_point_crossover(parent1, parent2)
        else:
            child1, child2 = parent1.copy(), parent2.copy()
        
        # Мутація
        child1 = self.random_reset_mutation(child1)
        child2 = self.random_reset_mutation(child2)
        
        new_population.append(child1)
        if len(new_population) < self.config.population_size:
            new_population.append(child2)
    
    self.population = np.array(new_population)

# Додаємо метод до класу
EvolutionaryAlgorithm.evolve_generation = evolve_generation

print("✅ Еволюція одного покоління додана")

In [ ]:
# Додаємо метод запуску алгоритму
def run(self, fitness_func, verbose: bool = True) -> dict:
    """Запуск еволюційного алгоритму"""
    start_time = time.time()
    
    # Ініціалізація
    self.population = self.initialize_population()
    
    if verbose:
        print(f"🚀 Запуск еволюційного алгоритму")
        print(self.config)
        print(f"Розмірність задачі: {self.problem_dim}")
        print(f"Обмеження: {self.bounds}")
    
    # Головний цикл еволюції
    for generation in range(self.config.generations):
        self.evolve_generation(fitness_func)
        
        if verbose and (generation + 1) % 50 == 0:
            best_fitness = self.history['best_fitness'][-1]
            avg_fitness = self.history['avg_fitness'][-1]
            print(f"Покоління {generation + 1:3d}: "
                  f"Найкращий = {best_fitness:.6f}, "
                  f"Середній = {avg_fitness:.6f}")
    
    execution_time = time.time() - start_time
    
    # Фінальні результати
    best_idx = np.argmin(self.fitness_values)
    best_individual = self.population[best_idx]
    best_fitness = self.fitness_values[best_idx]
    
    results = {
        'best_individual': best_individual,
        'best_fitness': best_fitness,
        'execution_time': execution_time,
        'history': self.history.copy()
    }
    
    if verbose:
        print(f"\\n✅ Еволюція завершена за {execution_time:.2f} секунд")
        print(f"🏆 Найкращий результат: {best_fitness:.6f}")
        print(f"🎯 Найкраща особина: {best_individual}")
    
    return results

# Додаємо метод до класу
EvolutionaryAlgorithm.run = run

print("✅ Основний еволюційний алгоритм завершено")

## 2. Задача 1: Функція Розенброка

Функція Розенброка - класична тестова функція для алгоритмів оптимізації:
$$f(x_1, x_2, ..., x_n) = \sum_{i=1}^{n-1} [100(x_{i+1} - x_i^2)^2 + (1 - x_i)^2]$$

Глобальний мінімум знаходиться в точці $(1, 1, ..., 1)$ з значенням $f = 0$.

In [ ]:
def rosenbrock_function(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> float:
    """Функція Розенброка для n-мірного простору"""
    x = np.asarray(x)
    if len(x.shape) == 0:  # Скаляр
        x = np.array([x])
    
    if len(x) == 1:
        return (a - x[0])**2
    
    result = 0.0
    for i in range(len(x) - 1):
        result += b * (x[i+1] - x[i]**2)**2 + (a - x[i])**2
    
    return result

# Тестуємо функцію
print("🧪 Тест функції Розенброка:")
print(f"f([1, 1]) = {rosenbrock_function([1, 1]):.6f} (повинно бути ≈ 0)")
print(f"f([0, 0]) = {rosenbrock_function([0, 0]):.6f}")
print(f"f([-1, 1]) = {rosenbrock_function([-1, 1]):.6f}")

print("✅ Функція Розенброка готова")

In [ ]:
# Візуалізуємо функцію Розенброка в 2D
x = np.linspace(-2, 2, 100)
y = np.linspace(-1, 3, 100)
X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)

for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rosenbrock_function([X[i, j], Y[i, j]])

plt.figure(figsize=(12, 5))

# Контурний графік
plt.subplot(1, 2, 1)
contour = plt.contour(X, Y, Z, levels=20, cmap='viridis')
plt.colorbar(contour)
plt.plot(1, 1, 'r*', markersize=15, label='Глобальний мінімум (1,1)')
plt.title('Функція Розенброка - Контури')
plt.xlabel('x₁')
plt.ylabel('x₂')
plt.legend()
plt.grid(True, alpha=0.3)

# 3D поверхня
ax = plt.subplot(1, 2, 2, projection='3d')
surf = ax.plot_surface(X, Y, Z, cmap='viridis', alpha=0.6)
ax.scatter([1], [1], [0], color='red', s=100, label='Глобальний мінімум')
plt.colorbar(surf, shrink=0.5)
ax.set_title('Функція Розенброка - 3D')
ax.set_xlabel('x₁')
ax.set_ylabel('x₂')
ax.set_zlabel('f(x₁, x₂)')

plt.tight_layout()
plt.show()

print("✅ Візуалізація функції Розенброка створена")

In [ ]:
# Конфігурації для експериментів з Розенброком
rosenbrock_configs = {
    'базова': EvolutionConfig(
        population_size=50,
        generations=300,
        crossover_rate=0.8,
        mutation_rate=0.1,
        elitism=2,
        selection_pressure=2.0
    ),
    'високий_мутація': EvolutionConfig(
        population_size=50,
        generations=300,
        crossover_rate=0.8,
        mutation_rate=0.2,  # Вища мутація
        elitism=2,
        selection_pressure=2.0
    ),
    'більша_популяція': EvolutionConfig(
        population_size=100,  # Більша популяція
        generations=200,      # Менше поколінь
        crossover_rate=0.8,
        mutation_rate=0.1,
        elitism=5,
        selection_pressure=2.0
    )
}

print("🔧 Налаштування експериментів для Розенброка:")
for name, config in rosenbrock_configs.items():
    print(f"\\n{name.upper()}:")
    print(f"  Популяція: {config.population_size}, Покоління: {config.generations}")
    print(f"  Схрещування: {config.crossover_rate}, Мутація: {config.mutation_rate}")
    print(f"  Елітизм: {config.elitism}, Тиск відбору: {config.selection_pressure}")

In [ ]:
# Запуск експериментів для функції Розенброка
print("🧬 Експерименти з функцією Розенброка (2D)")
print("="*60)

rosenbrock_results = {}
rosenbrock_dimension = 2
rosenbrock_bounds = (-5.0, 5.0)

for config_name, config in rosenbrock_configs.items():
    print(f"\\n🔬 Експеримент: {config_name.upper()}")
    print("-" * 40)
    
    # Створюємо та запускаємо еволюційний алгоритм
    ea = EvolutionaryAlgorithm(config, rosenbrock_dimension, rosenbrock_bounds)
    results = ea.run(rosenbrock_function, verbose=True)
    
    rosenbrock_results[config_name] = results
    
    print(f"\\n📊 Результати експерименту '{config_name}':")
    print(f"   Найкращий фітнес: {results['best_fitness']:.8f}")
    print(f"   Найкраща особина: [{results['best_individual'][0]:.4f}, {results['best_individual'][1]:.4f}]")
    print(f"   Час виконання: {results['execution_time']:.2f} сек")
    print(f"   Відстань до оптимуму: {np.linalg.norm(results['best_individual'] - np.array([1.0, 1.0])):.6f}")
    
print("\\n✅ Всі експерименти з Розенброком завершено")

## 3. Задача 2: Задача рюкзака

Класична задача оптимізації з дискретними змінними. Маємо рюкзак певної ваги та набір предметів з різними вагами та цінностями.

In [ ]:
class KnapsackProblem:
    """Задача рюкзака"""
    
    def __init__(self, items: List[Tuple[float, float]], capacity: float):
        """items: список кортежів (вага, цінність), capacity: максимальна вага"""
        self.items = items
        self.capacity = capacity
        self.n_items = len(items)
        self.weights = np.array([item[0] for item in items])
        self.values = np.array([item[1] for item in items])
        
        # Для порівняння - точний розв'язок методом динамічного програмування
        self.optimal_solution = self.solve_dp()
    
    def fitness(self, individual: np.ndarray) -> float:
        """Функція фітнесу для особини (бінарний вектор)"""
        # Перетворюємо до бінарного
        binary_individual = (individual > 0.5).astype(int)
        
        total_weight = np.sum(binary_individual * self.weights)
        total_value = np.sum(binary_individual * self.values)
        
        # Штраф за перевищення ваги
        if total_weight > self.capacity:
            penalty = (total_weight - self.capacity) * max(self.values)
            return -(total_value - penalty)  # Мінімізуємо негативну цінність
        
        return -total_value  # Мінімізуємо негативну цінність
    
    def decode_solution(self, individual: np.ndarray) -> dict:
        """Декодування розв'язку"""
        binary_individual = (individual > 0.5).astype(int)
        selected_items = np.where(binary_individual == 1)[0]
        
        total_weight = np.sum(binary_individual * self.weights)
        total_value = np.sum(binary_individual * self.values)
        
        return {
            'selected_items': selected_items.tolist(),
            'binary_solution': binary_individual,
            'total_weight': total_weight,
            'total_value': total_value,
            'is_feasible': total_weight <= self.capacity,
            'efficiency': total_value / total_weight if total_weight > 0 else 0
        }

print("✅ Клас KnapsackProblem створено")

In [ ]:
# Додаємо метод динамічного програмування до KnapsackProblem
def solve_dp(self) -> dict:
    """Точний розв'язок методом динамічного програмування"""
    n = len(self.items)
    W = int(self.capacity)
    
    # DP таблиця
    dp = np.zeros((n + 1, W + 1))
    
    for i in range(1, n + 1):
        weight, value = self.weights[i-1], self.values[i-1]
        for w in range(W + 1):
            if weight <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w-int(weight)] + value)
            else:
                dp[i][w] = dp[i-1][w]
    
    # Відновлення розв'язку
    selected = []
    w = W
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            selected.append(i-1)
            w -= int(self.weights[i-1])
    
    optimal_value = dp[n][W]
    optimal_weight = sum(self.weights[i] for i in selected)
    
    return {
        'selected_items': sorted(selected),
        'total_value': optimal_value,
        'total_weight': optimal_weight,
        'binary_solution': [(i in selected) for i in range(n)]
    }

# Додаємо метод до класу
KnapsackProblem.solve_dp = solve_dp

print("✅ Динамічне програмування додано до KnapsackProblem")

In [ ]:
# Генеруємо тестовий набір даних для рюкзака
def generate_knapsack_instance(n_items: int, capacity: float, seed: int = 42) -> KnapsackProblem:
    """Генерація випадкового екземпляру задачі рюкзака"""
    np.random.seed(seed)
    
    # Генеруємо ваги та цінності
    weights = np.random.uniform(1, capacity/4, n_items)  # Ваги від 1 до 1/4 ємності
    values = np.random.uniform(1, 100, n_items)          # Цінності від 1 до 100
    
    items = list(zip(weights, values))
    return KnapsackProblem(items, capacity)

# Створюємо тестовий екземпляр
knapsack = generate_knapsack_instance(n_items=30, capacity=50.0, seed=42)

print("🎒 Створено задачу рюкзака:")
print(f"   - Кількість предметів: {knapsack.n_items}")
print(f"   - Максимальна вага: {knapsack.capacity}")
print(f"   - Загальна вага всіх предметів: {sum(knapsack.weights):.2f}")
print(f"   - Загальна цінність всіх предметів: {sum(knapsack.values):.2f}")
print(f"   - Оптимальне значення (DP): {knapsack.optimal_solution['total_value']:.2f}")

# Показуємо деталі предметів
items_df = pd.DataFrame({
    'Предмет': range(knapsack.n_items),
    'Вага': knapsack.weights,
    'Цінність': knapsack.values,
    'Ефективність': knapsack.values / knapsack.weights
})

print("\\n📋 Топ-10 найефективніших предметів:")
print(items_df.nlargest(10, 'Ефективність')[['Предмет', 'Вага', 'Цінність', 'Ефективність']].round(2))

opt_sol = knapsack.optimal_solution
print(f"\\n🎯 Оптимальний розв'язок (DP):")
print(f"   Вибрані предмети: {opt_sol['selected_items']}")
print(f"   Загальна цінність: {opt_sol['total_value']:.2f}")
print(f"   Загальна вага: {opt_sol['total_weight']:.2f} / {knapsack.capacity}")

print("\\n✅ Задача рюкзака готова до оптимізації")

In [ ]:
# Конфігурації для експериментів з рюкзаком
knapsack_configs = {
    'базова': EvolutionConfig(
        population_size=60,
        generations=200,
        crossover_rate=0.8,
        mutation_rate=0.15,  # Трохи вища мутація для дискретних задач
        elitism=3,
        selection_pressure=2.0
    ),
    'інтенсивний_пошук': EvolutionConfig(
        population_size=40,
        generations=250,
        crossover_rate=0.9,   # Висока ймовірність схрещування
        mutation_rate=0.05,   # Низька мутація
        elitism=5,
        selection_pressure=3.0  # Високий тиск відбору
    ),
    'дослідження': EvolutionConfig(
        population_size=80,
        generations=150,
        crossover_rate=0.6,   # Нижче схрещування
        mutation_rate=0.25,   # Висока мутація
        elitism=2,
        selection_pressure=1.5  # Низький тиск відбору
    )
}

print("🔧 Налаштування експериментів для рюкзака:")
for name, config in knapsack_configs.items():
    print(f"\\n{name.upper()}:")
    print(f"  Популяція: {config.population_size}, Покоління: {config.generations}")
    print(f"  Схрещування: {config.crossover_rate}, Мутація: {config.mutation_rate}")
    print(f"  Елітизм: {config.elitism}, Тиск відбору: {config.selection_pressure}")

In [ ]:
# Запуск експериментів для задачі рюкзака
print("🎒 Експерименти з задачею рюкзака")
print("="*60)

knapsack_results = {}

for config_name, config in knapsack_configs.items():
    print(f"\\n🔬 Експеримент: {config_name.upper()}")
    print("-" * 40)
    
    # Створюємо та запускаємо еволюційний алгоритм
    ea = EvolutionaryAlgorithm(config, knapsack.n_items, (0.0, 1.0))
    results = ea.run(knapsack.fitness, verbose=True)
    
    # Декодуємо найкращий розв'язок
    decoded_solution = knapsack.decode_solution(results['best_individual'])
    results['decoded_solution'] = decoded_solution
    
    knapsack_results[config_name] = results
    
    print(f"\\n📊 Результати експерименту '{config_name}':")
    print(f"   Найкращий фітнес: {results['best_fitness']:.6f}")
    print(f"   Цінність розв'язку: {decoded_solution['total_value']:.2f}")
    print(f"   Вага розв'язку: {decoded_solution['total_weight']:.2f} / {knapsack.capacity}")
    print(f"   Допустимість: {'✅' if decoded_solution['is_feasible'] else '❌'}")
    print(f"   Кількість вибраних предметів: {len(decoded_solution['selected_items'])}")
    print(f"   Ефективність: {decoded_solution['efficiency']:.2f}")
    print(f"   Час виконання: {results['execution_time']:.2f} сек")
    
    # Порівняння з оптимальним розв'язком
    optimal_value = knapsack.optimal_solution['total_value']
    gap = ((optimal_value - decoded_solution['total_value']) / optimal_value) * 100
    print(f"   Відхилення від оптимуму: {gap:.2f}%")

print(f"\\n🎯 Порівняння з оптимальним розв'язком:")
print(f"   Оптимальна цінність: {knapsack.optimal_solution['total_value']:.2f}")
print(f"   Оптимальна вага: {knapsack.optimal_solution['total_weight']:.2f}")
print(f"   Оптимальні предмети: {knapsack.optimal_solution['selected_items']}")

print("\\n✅ Всі експерименти з рюкзаком завершено")

## 4. Аналіз та візуалізація результатів

Комплексний аналіз ефективності еволюційних операторів варіанту №7.

In [ ]:
# Функція для візуалізації результатів експериментів
def plot_evolution_history(results_dict, title: str):
    """Побудова графіків еволюції"""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'{title} - Аналіз конвергенції', fontsize=16, fontweight='bold')
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    
    # 1. Найкращий фітнес
    axes[0, 0].set_title('Еволюція найкращого фітнесу')
    axes[0, 0].set_xlabel('Покоління')
    axes[0, 0].set_ylabel('Найкращий фітнес')
    axes[0, 0].set_yscale('log')
    
    for i, (name, results) in enumerate(results_dict.items()):
        history = results['history']
        best_fitness = np.abs(history['best_fitness'])  # Для логарифмічного масштабу
        axes[0, 0].plot(best_fitness, label=name, color=colors[i % len(colors)], linewidth=2)
    
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Середній фітнес
    axes[0, 1].set_title('Еволюція середнього фітнесу')
    axes[0, 1].set_xlabel('Покоління')
    axes[0, 1].set_ylabel('Середній фітнес')
    
    for i, (name, results) in enumerate(results_dict.items()):
        history = results['history']
        axes[0, 1].plot(history['avg_fitness'], label=name, color=colors[i % len(colors)], linewidth=2)
    
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Різноманітність популяції
    axes[1, 0].set_title('Різноманітність популяції (Best vs Worst)')
    axes[1, 0].set_xlabel('Покоління')
    axes[1, 0].set_ylabel('Різниця (Worst - Best)')
    
    for i, (name, results) in enumerate(results_dict.items()):
        history = results['history']
        diversity = np.array(history['worst_fitness']) - np.array(history['best_fitness'])
        axes[1, 0].plot(diversity, label=name, color=colors[i % len(colors)], linewidth=2)
    
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Швидкість конвергенції
    axes[1, 1].set_title('Швидкість конвергенції')
    axes[1, 1].set_xlabel('Покоління')
    axes[1, 1].set_ylabel('Покращення фітнесу')
    
    for i, (name, results) in enumerate(results_dict.items()):
        history = results['history']
        best_fitness = np.array(history['best_fitness'])
        # Обчислюємо покращення (різниця з попереднім поколінням)
        improvement = np.abs(np.diff(best_fitness))
        # Згладжуємо за допомогою ковзного середнього
        window_size = min(10, len(improvement) // 4)
        if window_size > 1:
            improvement_smooth = np.convolve(improvement, np.ones(window_size)/window_size, mode='valid')
            x_smooth = range(window_size//2, len(improvement) - window_size//2 + 1)
            axes[1, 1].plot(x_smooth, improvement_smooth, label=name, color=colors[i % len(colors)], linewidth=2)
    
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_yscale('log')
    
    plt.tight_layout()
    plt.show()

print("✅ Функція візуалізації створена")

In [ ]:
# Візуалізуємо результати для обох задач
print("📈 Візуалізація результатів експериментів")

# Візуалізація для Розенброка
plot_evolution_history(rosenbrock_results, "Функція Розенброка")

# Візуалізація для рюкзака  
plot_evolution_history(knapsack_results, "Задача рюкзака")

In [ ]:
# Фінальний аналіз результатів
def comprehensive_analysis():
    """Комплексний аналіз всіх експериментів"""
    
    print("📋 КОМПЛЕКСНИЙ АНАЛІЗ РЕЗУЛЬТАТІВ ЛАБОРАТОРНОЇ РОБОТИ №3")
    print("="*80)
    
    # 1. Аналіз функції Розенброка
    print("\\n🌹 АНАЛІЗ ФУНКЦІЇ РОЗЕНБРОКА:")
    print("-" * 50)
    
    rosenbrock_summary = []
    for name, results in rosenbrock_results.items():
        distance_to_optimum = np.linalg.norm(results['best_individual'] - np.array([1.0, 1.0]))
        rosenbrock_summary.append({
            'Конфігурація': name,
            'Фітнес': results['best_fitness'],
            'Відстань до (1,1)': distance_to_optimum,
            'Час (сек)': results['execution_time'],
            'x₁': results['best_individual'][0],
            'x₂': results['best_individual'][1]
        })
    
    rosenbrock_df = pd.DataFrame(rosenbrock_summary)
    rosenbrock_df = rosenbrock_df.sort_values('Фітнес')
    print(rosenbrock_df.round(6).to_string(index=False))
    
    best_rosenbrock = rosenbrock_df.iloc[0]
    print(f"\\n🏆 Найкращий результат для Розенброка:")
    print(f"   Конфігурація: {best_rosenbrock['Конфігурація']}")
    print(f"   Фітнес: {best_rosenbrock['Фітнес']:.8f}")
    print(f"   Рішення: ({best_rosenbrock['x₁']:.6f}, {best_rosenbrock['x₂']:.6f})")
    print(f"   Відстань до оптимуму: {best_rosenbrock['Відстань до (1,1)']:.6f}")
    
    # 2. Аналіз задачі рюкзака
    print("\\n\\n🎒 АНАЛІЗ ЗАДАЧІ РЮКЗАКА:")
    print("-" * 50)
    
    knapsack_summary = []
    optimal_value = knapsack.optimal_solution['total_value']
    
    for name, results in knapsack_results.items():
        decoded = results['decoded_solution']
        gap = ((optimal_value - decoded['total_value']) / optimal_value) * 100
        knapsack_summary.append({
            'Конфігурація': name,
            'Цінність': decoded['total_value'],
            'Вага': decoded['total_weight'],
            'Предметів': len(decoded['selected_items']),
            'Ефективність': decoded['efficiency'],
            'Відхилення (%)': gap,
            'Час (сек)': results['execution_time']
        })
    
    knapsack_df = pd.DataFrame(knapsack_summary)
    knapsack_df = knapsack_df.sort_values('Цінність', ascending=False)
    print(knapsack_df.round(2).to_string(index=False))
    
    best_knapsack = knapsack_df.iloc[0]
    print(f"\\n🏆 Найкращий результат для рюкзака:")
    print(f"   Конфігурація: {best_knapsack['Конфігурація']}")
    print(f"   Цінність: {best_knapsack['Цінність']:.2f} (оптимум: {optimal_value:.2f})")
    print(f"   Відхилення від оптимуму: {best_knapsack['Відхилення (%)']:.2f}%")
    
    # 3. Аналіз операторів варіанту №7
    print("\\n\\n🧬 ВИСНОВКИ ПРО ЕВОЛЮЦІЙНІ ОПЕРАТОРИ (ВАРІАНТ №7):")
    print("-" * 60)
    
    print("✅ РАНЖУВАННЯ (Ranking Selection):")
    print("   • Забезпечує стабільну конвергенцію")
    print("   • Контролює тиск відбору")
    print("   • Запобігає передчасній конвергенції")
    
    print("\\n✅ БАГАТОТОЧКОВЕ СХРЕЩУВАННЯ (Multi-point Crossover):")
    print("   • Ефективне змішування генетичного матеріалу")
    print("   • Хороша експлорація простору рішень")
    print("   • Підходить для різних типів задач")
    
    print("\\n✅ ВИПАДКОВЕ СКИДАННЯ (Random Reset Mutation):")
    print("   • Сильна експлорація")
    print("   • Запобігає застряванню в локальних мінімумах")
    print("   • Ефективне для складних ландшафтів")
    
    # 4. Загальні висновки
    print("\\n\\n💡 ЗАГАЛЬНІ ВИСНОВКИ:")
    print("-" * 50)
    print("1. Еволюційні оператори варіанту №7 показали хорошу ефективність")
    print("2. Різні конфігурації краще підходять для різних типів задач")
    print("3. Параметри потребують адаптації під специфіку проблеми")
    
    if best_rosenbrock['Конфігурація'] == 'більша_популяція':
        print("4. Для неперервних задач ефективна більша популяція")
    elif best_rosenbrock['Конфігурація'] == 'високий_мутація':
        print("4. Для неперервних задач корисна висока мутація")
    else:
        print("4. Для неперервних задач достатні базові параметри")
    
    if best_knapsack['Конфігурація'] == 'інтенсивний_пошук':
        print("5. Для дискретних задач ефективний високий тиск відбору")
    elif best_knapsack['Конфігурація'] == 'дослідження':
        print("5. Для дискретних задач важлива інтенсивна експлорація")
    else:
        print("5. Для дискретних задач працюють збалансовані параметри")
    
    # 5. Статистика виконання
    total_time = sum(r['execution_time'] for r in rosenbrock_results.values()) + \\\n",
                 sum(r['execution_time'] for r in knapsack_results.values())
    
    print(f"\\n\\n⏱️ СТАТИСТИКА ВИКОНАННЯ:")
    print("-" * 40)
    print(f"Загальний час експериментів: {total_time:.2f} секунд")
    print(f"Кількість експериментів: {len(rosenbrock_results) + len(knapsack_results)}")
    print(f"Середній час на експеримент: {total_time / (len(rosenbrock_results) + len(knapsack_results)):.2f} сек")
    print(f"Відповідає вимогам Colab: {'✅' if total_time < 1800 else '⚠️'}")
    
    return {
        'rosenbrock_best': best_rosenbrock,
        'knapsack_best': best_knapsack,
        'total_time': total_time
    }

# Виконуємо комплексний аналіз
final_results = comprehensive_analysis()

print("\\n" + "="*80)
print("🎉 ЛАБОРАТОРНА РОБОТА №3 ЗАВЕРШЕНА УСПІШНО!")
print("="*80)
print("\\n📊 Результати експериментів готові для звіту")
print("🔬 Проведено аналіз ефективності операторів варіанту №7")
print("📈 Створено всі необхідні візуалізації")
print("⏱️ Час виконання оптимізовано для Google Colab")
print("\\n🎯 Готово для оформлення висновків про еволюційні алгоритми!")